# VadCLIP Prompt Text-Space Scoring And Weighting

Notebook n?y ch?m ?i?m c?c prompt bank `5x` v? `10x` trong text space c?a CLIP/VadCLIP. M?c ti?u hi?n t?i kh?ng ch? l? ch?n m?t prompt ??i di?n, m? c?n t?o **tr?ng s? cho t?ng prompt** ?? d?ng trong weighted multi-prompt training.

Score m?i prompt g?m ba th?nh ph?n:

- `anchor_similarity`: prompt g?n embedding c?a t?n class g?c.
- `intra_prompt_similarity`: prompt g?n c?c prompt kh?c c?ng class.
- `inter_prompt_similarity`: prompt xa prompt c?a class kh?c.

M?c ??nh notebook d?ng `raw_clip_encode_text` v? `scoring_prompt_mode = prototype_only`, ngh?a l? prompt ???c encode ??ng n?i dung g?c, kh?ng t? th?m class name ph?a tr??c. C?ch n?y gi?p ki?m tra tr?c ti?p nghi v?n: prompt c? th?t s? g?n class name hay ?ang b? l?ch kh?i class anchor.


In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
else:
    NOTEBOOK_DIR = Path.cwd()
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR

SCRIPT_PATH = PROJECT_ROOT / 'code' / 'select_representative_class_prompts.py'
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
MODEL_PATH = PROJECT_ROOT / 'model_ucf.pth'

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(SCRIPT_PATH)
if not SRC_DIR.exists():
    raise FileNotFoundError(SRC_DIR)
if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SCRIPT_PATH:', SCRIPT_PATH)
print('MODEL_PATH:', MODEL_PATH)

## Install Dependencies

In [ ]:
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'scikit-learn', 'scipy', 'matplotlib'],
        check=True,
    )
print('Dependencies ready')

## Select Representative Prompts

In [ ]:
# Input: all existing 5x/10x prompt banks. Excluding generated selected/representative files.
prompt_jsons = []
for pattern in ['ucf_manual_class_prompts_5x*.json', 'ucf_manual_class_prompts_10x*.json']:
    prompt_jsons.extend((PROJECT_ROOT / 'code').glob(pattern))
prompt_jsons = sorted({p.resolve() for p in prompt_jsons if 'selected' not in p.stem and 'representative' not in p.stem})

if not prompt_jsons:
    raise FileNotFoundError('No prompt bank files found in code/ucf_manual_class_prompts_5x*.json or 10x*.json')

# The script writes the main weighted output from the first prompt JSON.
# Keep v5 first because it is the current 5-prompt bank we want to train next.
MAIN_WEIGHTED_SOURCE = (PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v5.json').resolve()
if MAIN_WEIGHTED_SOURCE in prompt_jsons:
    prompt_jsons = [MAIN_WEIGHTED_SOURCE] + [p for p in prompt_jsons if p != MAIN_WEIGHTED_SOURCE]

print('Prompt JSON inputs:')
for p in prompt_jsons:
    marker = '  <-- main weighted output source' if p == prompt_jsons[0] else ''
    print(' -', p, marker)

OUTPUT_DIR = PROJECT_ROOT / 'code' / 'vadclip_representative_prompt_selection'
SELECTED_OUTPUT_JSON = PROJECT_ROOT / 'code' / 'ucf_selected_representative_class_prompts.json'
WEIGHTED_OUTPUT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'

# Score weights: class-name anchor is intentionally strongest.
W_ANCHOR = 0.50
W_INTRA = 0.30
W_INTER = 0.20
INTER_MODE = 'max'  # max | top3_mean | mean
MAIN_SPACE = 'raw_clip_encode_text'
SCORING_PROMPT_MODE = 'prototype_only'  # prototype_only measures raw prompt vs class name directly.
PROMPT_WEIGHT_TEMPERATURE = 0.20

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--model-path', str(MODEL_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--selected-output-json', str(SELECTED_OUTPUT_JSON),
    '--weighted-output-json', str(WEIGHTED_OUTPUT_JSON),
    '--w-anchor', str(W_ANCHOR),
    '--w-intra', str(W_INTRA),
    '--w-inter', str(W_INTER),
    '--inter-mode', INTER_MODE,
    '--main-space', MAIN_SPACE,
    '--scoring-prompt-mode', SCORING_PROMPT_MODE,
    '--prompt-weight-temperature', str(PROMPT_WEIGHT_TEMPERATURE),
    '--spaces', 'vadclip_encode_textprompt', 'raw_clip_encode_text',
    '--prompt-jsons', *[str(p) for p in prompt_jsons],
]

print('\nRunning:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

print('\nMain selected prompt JSON:')
print(SELECTED_OUTPUT_JSON)
print('\nMain weighted prompt JSON:')
print(WEIGHTED_OUTPUT_JSON)
print('\nFull output directory:')
print(OUTPUT_DIR)


## Preview Selected Prompt Bank

In [ ]:
import csv
import json

payload = json.loads(SELECTED_OUTPUT_JSON.read_text(encoding='utf-8'))
weighted_payload = json.loads(WEIGHTED_OUTPUT_JSON.read_text(encoding='utf-8'))
print('Selected prompt bank:', SELECTED_OUTPUT_JSON)
print('Selection method:', payload['metadata'].get('selection_method'))
print('Embedding space:', payload['metadata'].get('embedding_space'))
print('Scoring prompt mode:', payload['metadata'].get('scoring_prompt_mode'))
print('\nSelected prompts:')
for class_name, prompts in payload['classes'].items():
    detail = payload.get('selection_details', {}).get(class_name, {})
    print(
        f"{class_name:<13} score={detail.get('selection_score', 0):+.4f} "
        f"anchor_sim={detail.get('anchor_similarity', 0):.4f} "
        f"margin={detail.get('anchor_margin', 0):+.4f} "
        f"source={detail.get('source_stem', '')}#{detail.get('prompt_index', '')} | {prompts[0]}"
    )

print('\nWeighted prompt preview from main weighted JSON:')
for class_name, prompts in weighted_payload['classes'].items():
    weights = weighted_payload['prompt_weights'][class_name]
    pairs = sorted(zip(weights, prompts), reverse=True)
    print(f'\n{class_name}:')
    for weight, prompt in pairs:
        print(f'  weight={weight:.4f} | {prompt}')

sim_csv = OUTPUT_DIR / MAIN_SPACE / 'prompt_to_class_anchor_similarity.csv'
print('\nLowest prompt-to-own-class similarity rows:')
with sim_csv.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
rows = sorted(rows, key=lambda r: float(r['own_class_similarity']))
for row in rows[:30]:
    print(
        f"{row['class_name']:<13} {row['source_stem']}#{row['prompt_index']:<2} "
        f"own={float(row['own_class_similarity']):.4f} "
        f"best_other={row['best_other_anchor_class']}:{float(row['best_other_anchor_similarity']):.4f} "
        f"margin={float(row['anchor_margin']):+.4f} | {row['raw_prompt_text']}"
    )
